# Growth and inflation quadrants

This notebook asks how a fixed set of liquid cross-asset proxies behaves after four
combinations of real-time industrial-production growth and consumer-price inflation.
It is a descriptive regime study, not a causal model, forecast, or trading strategy.

The hypothesis and thresholds were recorded in `studies/README.md` before live
execution. The committed notebook contains no saved provider values or outputs.
Running it retrieves live Alpha Vantage and FRED or ALFRED data into a temporary
directory, builds every table and figure in memory, and removes the raw responses.

**Primary protocol.** A decision is the last common ETF trading session in a complete
calendar month. Information must have been available at least one calendar day before
that close. The unit of analysis is one decision month. The primary horizon is the
next calendar month. The four primary estimands are average two-factor contrasts:
high minus lower inflation for `GLD` and `DBC`, and contracting minus expanding
growth for `SPY` and `IEF`. The other factor and its interaction remain in each model.
One-month simultaneous intervals control this four-contrast family; longer horizons,
threshold sweeps, and latest-revised comparisons are exploratory.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from studies._support import (
    CORE_SYMBOLS,
    STUDY_START,
    acquire_latest_series,
    acquire_monthly_prices,
    acquire_vintage_histories,
    assert_component_periods_match,
    build_point_in_time_levels,
    classification_transition_table,
    compare_statistics,
    configure_plots,
    factorial_contrast_statistics,
    familywise_primary_intervals,
    feature_provenance_summary,
    forward_labels,
    latest_revised_year_over_year,
    momentum_baseline,
    open_live_session,
    plot_coverage,
    plot_feature_comparison,
    plot_normalized_prices,
    plot_regime_contrasts,
    plot_regime_distributions,
    plot_regime_means,
    plot_regime_timeline,
    plot_revision_gap,
    plot_sample_sizes,
    plot_sensitivity_heatmap,
    point_in_time_year_over_year,
    regime_statistics,
    regime_style,
    simultaneous_interval_family,
    study_run_manifest,
    temporal_factorial_stability,
    unconditional_statistics,
    validate_study_outputs,
)

configure_plots()
pd.set_option("display.max_columns", 20)
session = open_live_session()

## Pre-analysis protocol and universe

`SPY`, `IEF`, `GLD`, `DBC`, and `UUP` represent equities, intermediate Treasuries,
gold, broad commodities, and the United States dollar. Adjusted closes account for
distributions and splits according to the provider's adjustment method. Month-end
sampling reduces unequal holiday calendars to one common decision frequency.

`CPIAUCSL` measures the consumer price level and `INDPRO` measures industrial output.
Both are revised. Their values therefore need an availability interval as well as an
observation period. The one-day operational lag is conservative: a release that
becomes available on a decision date is not used until the next calendar day.

Adjusted ETF closes are total-return proxies, not executable transaction prices. The
code acquires one year of price history before the analysis window so the twelve-month
momentum baseline exists on the first eligible outcome date. It rejects missing or
duplicate calendar months and common closes too far from month end. Raw responses and
normalized objects also round-trip through a temporary DuckDB database. That storage
check is part of the end-to-end path and disappears when the session closes.

In [ ]:
price_history, market_provenance = acquire_monthly_prices(session, CORE_SYMBOLS)
prices = price_history.loc[STUDY_START:]
series_ids = ("CPIAUCSL", "INDPRO")
histories = acquire_vintage_histories(session, series_ids, prices.index)
latest = acquire_latest_series(session, series_ids)
staleness = {series_id: pd.Timedelta(days=62) for series_id in series_ids}
point_in_time = build_point_in_time_levels(
    histories,
    prices.index,
    staleness,
)
feature_provenance = feature_provenance_summary(point_in_time)
manifest = study_run_manifest(
    prices,
    series_ids=series_ids,
    thresholds="growth=0%; inflation=3%",
    staleness=staleness,
)
display(manifest, market_provenance, feature_provenance)

## Coverage is part of the result

A long price history does not guarantee a complete joint panel. ETF inception dates,
missing sessions, release schedules, explicit missing macro observations, and the
staleness ceiling can all remove decisions. The first panel below shows provider
history depth. The second reports the share of decision dates with an admissible
macro version. Sparse coverage is not filled or silently replaced with an older
nonmissing observation.

Three dates have different jobs. The **observation period** says which month the
level describes. `available_from` and `available_through` define the source version's
historical validity interval. `retrieved_at` records when this execution obtained the
response. A row is admissible only when the lagged decision cutoff lies inside its
availability interval and its observation age is below the declared ceiling.

In [ ]:
figure, _ = plot_coverage(market_provenance, feature_provenance)
plt.show()
plt.close(figure)

## From source levels to point-in-time features

For decision date \(d\), the code selects one complete vintage snapshot known at the
information cutoff. If \(x_{d,t}\) is the level for its newest admissible source month
\(t\), the feature is \(100(x_{d,t}/x_{d,t-12}-1)\). Numerator and denominator come
from one coherent vintage and exact observation months, not values captured on two
different decision dates. This matters when benchmark revisions rebase a level path.

Component provenance records expected and actual periods, availability intervals,
source definition fields, and retrieval time. The latest-revised comparison preserves
those exact component periods and swaps in today's values. It isolates revision
substitution rather than inventing an earlier release date or changing the window.

The plot is diagnostic. A visible gap shows where retrospective history differs from
the feature available at the time; agreement does not prove that the timing policy was
unnecessary.

In [ ]:
point_yoy_result = point_in_time_year_over_year(histories, point_in_time)
latest_yoy_result = latest_revised_year_over_year(point_in_time, latest)
point_yoy = point_yoy_result.frame
latest_yoy = latest_yoy_result.frame
point_features = point_yoy.rename(
    columns={"CPIAUCSL": "CPI inflation", "INDPRO": "Industrial production growth"}
)
latest_features = latest_yoy.set_axis(point_features.columns, axis="columns")
figure, _ = plot_feature_comparison(
    point_features,
    latest_features,
    tuple(point_features.columns),
)
plt.show()
plt.close(figure)

## Hypothesis and regime construction

The primary rules are fixed before outcome analysis. Inflation is high at or above
three percent. Industrial-production growth is contracting at or below zero. The
Cartesian product yields four named quadrants. These boundaries are transparent and
interpretable, but they are not natural laws. Later sensitivity analysis evaluates a
complete nearby grid rather than choosing the threshold that looks best.

`GLD` and `DBC` are the focal inflation assets; `SPY` and `IEF` are the focal growth
assets. “Focal” does not claim that their effects rank above every other ETF. For each
outcome, an effect-coded saturated model uses growth, inflation, and their interaction.
The inflation coefficient averages the high-minus-lower simple effect equally across
expanding and contracting states; the growth coefficient averages across inflation
states. This factorial estimand prevents unequal composition of the other macro
dimension from masquerading as the focal effect.

The timeline and phase portrait expose persistence and sparse quadrants before any
outcome statistic is read. Adjacent months in one uninterrupted state form one episode,
so a month count is not an independent-event count.

In [ ]:
def growth_inflation_regime(
    features: pd.DataFrame,
    *,
    inflation_boundary: float = 3.0,
    growth_boundary: float = 0.0,
) -> pd.Series:
    inflation = features["CPIAUCSL"]
    growth = features["INDPRO"]
    valid = inflation.notna() & growth.notna()
    state = pd.Series(pd.NA, index=features.index, dtype="string")
    state.loc[valid & growth.gt(growth_boundary) & inflation.lt(inflation_boundary)] = (
        "expanding / lower inflation"
    )
    state.loc[valid & growth.gt(growth_boundary) & inflation.ge(inflation_boundary)] = (
        "expanding / high inflation"
    )
    state.loc[valid & growth.le(growth_boundary) & inflation.lt(inflation_boundary)] = (
        "contracting / lower inflation"
    )
    state.loc[valid & growth.le(growth_boundary) & inflation.ge(inflation_boundary)] = (
        "contracting / high inflation"
    )
    return state

def inflation_state(
    features: pd.DataFrame,
    boundary: float = 3.0,
) -> pd.Series:
    values = features["CPIAUCSL"]
    valid = features[["CPIAUCSL", "INDPRO"]].notna().all(axis=1)
    state = pd.Series(pd.NA, index=features.index, dtype="string")
    state.loc[valid & values.lt(boundary)] = "lower inflation"
    state.loc[valid & values.ge(boundary)] = "high inflation"
    return state

def growth_state(
    features: pd.DataFrame,
    boundary: float = 0.0,
) -> pd.Series:
    values = features["INDPRO"]
    valid = features[["CPIAUCSL", "INDPRO"]].notna().all(axis=1)
    state = pd.Series(pd.NA, index=features.index, dtype="string")
    state.loc[valid & values.gt(boundary)] = "expanding"
    state.loc[valid & values.le(boundary)] = "contracting"
    return state

point_regimes = growth_inflation_regime(point_yoy)
latest_regimes = growth_inflation_regime(latest_yoy)
point_inflation_states = inflation_state(point_yoy)
point_growth_states = growth_state(point_yoy)
display(point_regimes.value_counts(dropna=False).rename("decision count"))
figure, _ = plot_regime_timeline(
    point_yoy["CPIAUCSL"],
    point_regimes,
    title="Real-time inflation with growth-inflation quadrant markers",
    ylabel="Year-over-year percent change",
    boundaries=(3.0,),
)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(9, 6.5))
for regime in (
    "expanding / lower inflation",
    "expanding / high inflation",
    "contracting / lower inflation",
    "contracting / high inflation",
):
    color, marker = regime_style(regime)
    selected = point_regimes.eq(regime).fillna(False)
    axis.scatter(
        point_yoy.loc[selected, "INDPRO"],
        point_yoy.loc[selected, "CPIAUCSL"],
        label=regime,
        marker=marker,
        color=color,
        alpha=0.75,
    )
axis.axvline(0, color="#333333", linestyle="--", linewidth=0.9)
axis.axhline(3, color="#333333", linestyle="--", linewidth=0.9)
axis.set(
    title="Real-time growth-inflation phase portrait",
    xlabel="Industrial-production year-over-year growth (percent)",
    ylabel="Consumer-price year-over-year inflation (percent)",
)
axis.legend(ncol=2)
figure.tight_layout()
plt.show()
plt.close(figure)

## Future labels and conventional baselines

The price panel is passed to `persistra.research.forward_returns`, which returns a
dedicated label object and records the end date of every horizon. No forward return
enters the feature frame. The final one, three, or twelve rows remain missing when a
complete future horizon is unavailable.

Two baselines discourage regime storytelling. The unconditional table asks whether
a regime adds information beyond the same macro-eligible sample. The trailing
twelve-month price momentum split is a conventional asset-only time-series comparison
using information observable at the same decision date. The normalized-price plot
provides historical context but is not a simulated portfolio.

For price \(P_d\), an \(h\)-month label is \(R_{d,h}=P_{d+h}/P_d-1\). Each label
stores its actual ending close. Live checks require its calendar period to be exactly
\(h\) months after the start, so a missing month cannot silently lengthen the outcome.
Terminal labels remain missing rather than using a shorter horizon.

In [ ]:
labels = forward_labels(prices)
eligible = point_regimes.notna()
unconditional = unconditional_statistics(labels, eligible=eligible)
momentum = momentum_baseline(price_history, labels, eligible=eligible)
display(unconditional, momentum)
figure, _ = plot_normalized_prices(prices)
plt.show()
plt.close(figure)

## Conditional summaries and uncertainty

Each descriptive row reports observed count, outcome-eligible episode count, coverage,
mean simple return, horizon-return standard deviation, positive-return share, and a
pointwise two-sided interval. One-month annualized volatility is a separate column.
The Bartlett-kernel heteroskedasticity-and-autocorrelation-consistent (HAC) bandwidth
is the larger of the overlap floor \(h-1\) and a predeclared automatic time-series
rule. It can address covariance beyond mechanical
overlap, but remains a normal large-sample approximation.

The one-month summary also exercises Persistra's regime summarizer, including its
episode-aware drawdown calculation. The four primary one-month factorial contrasts
receive Bonferroni simultaneous intervals. Gray means only that a result falls below
the display rule of twelve outcomes per side, six per factorial cell, and two
outcome-eligible episodes per side; it does not guarantee reliable inference.
Secondary horizons and sensitivity families remain exploratory. First/second-half and
leave-one-episode-out tables expose fragility without redefining the primary estimate.

In [ ]:
point_statistics = regime_statistics(labels, point_regimes)
latest_statistics = regime_statistics(labels, latest_regimes)
inflation_contrasts = factorial_contrast_statistics(
    labels,
    point_inflation_states,
    point_growth_states,
    treated="high inflation",
    reference="lower inflation",
    adjustment_levels=("contracting", "expanding"),
    assets=("GLD", "DBC"),
)
growth_contrasts = factorial_contrast_statistics(
    labels,
    point_growth_states,
    point_inflation_states,
    treated="contracting",
    reference="expanding",
    adjustment_levels=("high inflation", "lower inflation"),
    assets=("SPY", "IEF"),
)
primary_contrasts = familywise_primary_intervals(
    pd.concat([inflation_contrasts, growth_contrasts], ignore_index=True)
)
display(point_statistics, primary_contrasts)
figure, _ = plot_regime_means(
    point_statistics,
    title="Growth-inflation quadrants and one-month outcomes",
)
plt.show()
plt.close(figure)

figure, _ = plot_regime_contrasts(
    primary_contrasts,
    title="Predeclared one-month boundary contrasts",
)
plt.show()
plt.close(figure)

## Distributions, episodes, and sample size

Conditional means can be dominated by a few crisis months. Box plots retain the
middle spread and skew without displaying every raw observation, while the count plot
makes rare quadrants unmistakable. Neither plot makes months independent: adjacent
decisions can belong to the same macro episode, and the same shock can influence
several horizons and assets.

In [ ]:
figure, _ = plot_regime_distributions(
    labels[1],
    point_regimes,
    assets=("SPY", "IEF", "GLD", "DBC"),
    title="One-month outcome distributions by quadrant",
)
plt.show()
plt.close(figure)

figure, _ = plot_sample_sizes(point_statistics)
plt.show()
plt.close(figure)

## Sensitivity without threshold shopping

Two one-dimensional sweeps preserve the factorial estimand. Inflation boundaries of
2.5, 3.0, and 3.5 percent are evaluated for `GLD` and `DBC` while the primary growth
factor remains in the model. Growth boundaries of -1, 0, and 1 percent are evaluated
for `SPY` and `IEF` while the primary inflation factor remains. This is not a Cartesian
search for the best diagonal quadrant.

The long table retains counts, outcome-eligible episodes, HAC errors, and intervals.
One exploratory Bonferroni family covers every displayed threshold-asset effect.
Cells below the minimum-data display rule are explicitly marked and masked in the
two feature-specific heatmaps; they are not zero effects. Separate panels avoid
implying that nonfocal feature-asset pairs were estimated. Visual stability is a
robustness diagnostic, not a new specification-selection rule.

In [ ]:
sensitivity_rows = []
for boundary in (2.5, 3.0, 3.5):
    table = factorial_contrast_statistics(
        {1: labels[1]},
        inflation_state(point_yoy, boundary),
        point_growth_states,
        treated="high inflation",
        reference="lower inflation",
        adjustment_levels=("contracting", "expanding"),
        assets=("GLD", "DBC"),
    )
    table.insert(0, "boundary", boundary)
    table.insert(0, "feature", "inflation")
    sensitivity_rows.append(table)
for boundary in (-1.0, 0.0, 1.0):
    table = factorial_contrast_statistics(
        {1: labels[1]},
        growth_state(point_yoy, boundary),
        point_inflation_states,
        treated="contracting",
        reference="expanding",
        adjustment_levels=("high inflation", "lower inflation"),
        assets=("SPY", "IEF"),
    )
    table.insert(0, "boundary", boundary)
    table.insert(0, "feature", "growth")
    sensitivity_rows.append(table)
sensitivity_table = simultaneous_interval_family(
    pd.concat(sensitivity_rows, ignore_index=True),
    family_id="growth-inflation threshold family",
)
inflation_rows = sensitivity_table.loc[sensitivity_table["feature"].eq("inflation")]
inflation_sensitivity = inflation_rows.pivot(
    index="boundary", columns="asset", values="mean_difference"
).where(
    inflation_rows.pivot(
        index="boundary", columns="asset", values="meets_display_threshold"
    )
)
growth_rows = sensitivity_table.loc[sensitivity_table["feature"].eq("growth")]
growth_sensitivity = growth_rows.pivot(
    index="boundary", columns="asset", values="mean_difference"
).where(
    growth_rows.pivot(
        index="boundary", columns="asset", values="meets_display_threshold"
    )
)
display(sensitivity_table, inflation_sensitivity, growth_sensitivity)
figure, _ = plot_sensitivity_heatmap(
    inflation_sensitivity,
    title="Inflation-boundary effects for focal inflation assets",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

figure, _ = plot_sensitivity_heatmap(
    growth_sensitivity,
    title="Growth-boundary effects for focal growth assets",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

## Latest-revised bias diagnostic

The latest-revised classification is deliberately wrong for historical decisions.
Comparing it with the point-in-time summary shows whether revisions change membership
or conditional means. The revision-gap plot stays in
the diagnostic layer: future revisions never become features in the primary analysis.

A small difference is a finding about these series and dates, not permission to ignore
availability in other research.

The transition table includes an explicit unclassified state. It separates boundary
crossings from dates lost to component availability. The conditional-summary
comparison is therefore a total revision-plus-availability diagnostic; it is not the
primary factorial estimate. The stability table belongs to robustness: it reports
first/second-half and leave-one-episode-out estimates using only real-time regimes.

In [ ]:
revision_comparison = compare_statistics(point_statistics, latest_statistics)
classification_changes = classification_transition_table(
    point_regimes,
    latest_regimes,
)
inflation_stability = temporal_factorial_stability(
    labels,
    point_inflation_states,
    point_growth_states,
    treated="high inflation",
    reference="lower inflation",
    adjustment_levels=("contracting", "expanding"),
    assets=("GLD", "DBC"),
)
growth_stability = temporal_factorial_stability(
    labels,
    point_growth_states,
    point_inflation_states,
    treated="contracting",
    reference="expanding",
    adjustment_levels=("high inflation", "lower inflation"),
    assets=("SPY", "IEF"),
)
factorial_stability = pd.concat(
    [inflation_stability, growth_stability], ignore_index=True
)
display(revision_comparison, classification_changes, factorial_stability)
figure, _ = plot_revision_gap(
    point_yoy["CPIAUCSL"],
    latest_yoy["CPIAUCSL"],
    title="Consumer-price inflation revision substitution gap",
)
plt.show()
plt.close(figure)

## Limitations and adversarial checks

The ETF universe is selected with current knowledge and begins after every core proxy
exists. It embeds survivorship and product-design choices. Adjusted closes omit
transaction costs, taxes, spreads, and executable decision timing. Macro staleness and
monthly sampling simplify intramonth release mechanics. Regime observations cluster,
confidence intervals are approximate, and many assets, horizons, quadrants, and
sensitivity cells create multiple-testing risk.

The final code asserts alignment, provenance completeness, positive prices, separate
labels, multiple regimes, nonempty summaries, and finite observed means. These checks
protect mechanics, not the economic hypothesis.

Current FRED definition metadata are attached to historical vintage rows, so the
notebook cannot prove that every old benchmark definition was unchanged. Within-vintage
ratios avoid cross-vintage level arithmetic, but benchmark changes remain part of the
diagnostic. See the [ALFRED real-time-period guide](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html),
[Alpha Vantage API documentation](https://www.alphavantage.co/documentation/), and
Newey and West's [HAC covariance paper](https://doi.org/10.2307/1913610).

In [ ]:
audit = validate_study_outputs(
    prices,
    point_in_time,
    labels,
    point_regimes,
    point_statistics,
    expected_regimes=frozenset(
        {
            "expanding / lower inflation",
            "expanding / high inflation",
            "contracting / lower inflation",
            "contracting / high inflation",
        }
    ),
    transformed=point_yoy_result,
)
assert_component_periods_match(point_yoy_result, latest_yoy_result)
assert set(point_in_time.frame.columns).isdisjoint(labels[1].frame.columns)
matched = point_in_time.provenance["available_from"].notna()
assert point_in_time.provenance.loc[matched, "available_from"].le(
    point_in_time.provenance.loc[matched, "decision_date"] - pd.Timedelta(days=1)
).all()
display(audit)
session.close()

## Interpretation after execution

Read coverage and counts before means, then compare intervals, distributions, the
unconditional and momentum baselines, the complete sensitivity grid, and the
latest-revised diagnostic. Retain null, unstable, and contrary results. Any observed
association describes this fixed sample; it is not causal evidence or proof of a
profitable allocation rule.